# Optimization Methods Programming Assignment
## Solving a Real-World Optimization Problem

**Topic:** Medical Delivery Route Optimization using Exact Dynamic Programming and a Heuristic Local Search approach.

### Problem statement
A central depot in Galle must deliver medicines to multiple health points while minimizing the total travel distance. This is modeled as a Traveling Salesman Problem (TSP).

In [1]:
import math, itertools, time
import pandas as pd
import matplotlib.pyplot as plt

locations = [('Depot - Galle General Hospital', 0.0, 0.0), ('Hikkaduwa Pharmacy', 16, 18), ('Ambalangoda Medical Center', 28, 30), ('Elpitiya Clinic', 24, 10), ('Baddegama Pharmacy', 12, 8), ('Akuressa Rural Hospital', -8, 18), ('Imaduwa Medical Point', 4, -10), ('Koggala Pharmacy', 10, -18), ('Weligama Health Store', 22, -24), ('Deniyaya Clinic', -18, 12), ('Yakkalamulla Pharmacy', -6, -6), ('Udugama Rural Dispensary', -20, 2), ('Karapitiya Sub Depot', 2, 4)]
coords = [(x, y) for _, x, y in locations]
D = [[0 if i == j else round(math.hypot(coords[i][0]-coords[j][0], coords[i][1]-coords[j][1]), 2) for j in range(len(coords))] for i in range(len(coords))]
pd.DataFrame(locations, columns=["Location", "x", "y"])

,Location,x,y
0,Depot - Galle General Hospital,0.0,0.0
1,Hikkaduwa Pharmacy,16.0,18.0
2,Ambalangoda Medical Center,28.0,30.0
3,Elpitiya Clinic,24.0,10.0
4,Baddegama Pharmacy,12.0,8.0
5,Akuressa Rural Hospital,-8.0,18.0
6,Imaduwa Medical Point,4.0,-10.0
7,Koggala Pharmacy,10.0,-18.0
8,Weligama Health Store,22.0,-24.0
9,Deniyaya Clinic,-18.0,12.0


### Exact method: Held-Karp Dynamic Programming
This exact method guarantees the optimal route, but its computational complexity grows exponentially.

In [2]:

def held_karp(dist_matrix):
    n = len(dist_matrix)
    C = {}
    parent = {}
    for k in range(1, n):
        mask = 1 << (k - 1)
        C[(mask, k)] = dist_matrix[0][k]
        parent[(mask, k)] = 0
    for subset_size in range(2, n):
        for subset in itertools.combinations(range(1, n), subset_size):
            bits = 0
            for b in subset:
                bits |= 1 << (b - 1)
            for k in subset:
                prev = bits & ~(1 << (k - 1))
                best = float('inf')
                best_prev = None
                for m in subset:
                    if m == k:
                        continue
                    val = C[(prev, m)] + dist_matrix[m][k]
                    if val < best:
                        best = val
                        best_prev = m
                C[(bits, k)] = best
                parent[(bits, k)] = best_prev
    bits = (1 << (n - 1)) - 1
    best = float('inf')
    last = None
    for k in range(1, n):
        val = C[(bits, k)] + dist_matrix[k][0]
        if val < best:
            best = val
            last = k
    seq = []
    subset = bits
    k = last
    while subset:
        seq.append(k)
        pk = parent[(subset, k)]
        subset &= ~(1 << (k - 1))
        k = pk
        if k == 0:
            break
    route = [0] + list(reversed(seq)) + [0]
    return round(best, 2), route

def route_cost(route, dist_matrix):
    return round(sum(dist_matrix[route[i]][route[i+1]] for i in range(len(route)-1)), 2)


### Heuristic method: Nearest Neighbor + 2-opt
This method is faster and scales better, but does not guarantee optimality.

In [3]:

def nearest_neighbor(dist_matrix):
    unvisited = set(range(1, len(dist_matrix)))
    route = [0]
    current = 0
    while unvisited:
        nxt = min(unvisited, key=lambda j: dist_matrix[current][j])
        route.append(nxt)
        unvisited.remove(nxt)
        current = nxt
    route.append(0)
    return route

def two_opt(route, dist_matrix):
    best = route[:]
    improved = True
    while improved:
        improved = False
        best_cost = route_cost(best, dist_matrix)
        for i in range(1, len(best) - 2):
            for j in range(i + 1, len(best) - 1):
                if j - i == 1:
                    continue
                candidate = best[:i] + best[i:j][::-1] + best[j:]
                cand_cost = route_cost(candidate, dist_matrix)
                if cand_cost < best_cost:
                    best, best_cost = candidate, cand_cost
                    improved = True
    return best


In [4]:

t0 = time.perf_counter()
exact_cost, exact_route = held_karp(D)
exact_time = time.perf_counter() - t0

t0 = time.perf_counter()
nn_route = nearest_neighbor(D)
heuristic_route = two_opt(nn_route, D)
heuristic_time = time.perf_counter() - t0
heuristic_cost = route_cost(heuristic_route, D)

print('Exact cost:', exact_cost, 'route:', exact_route, 'time:', round(exact_time, 6))
print('Heuristic cost:', heuristic_cost, 'route:', heuristic_route, 'time:', round(heuristic_time, 6))


Exact cost: 189.31 route: [0, 12, 4, 1, 2, 3, 8, 7, 6, 10, 11, 9, 5, 0] time: 0.039122
Heuristic cost: 193.51 route: [0, 12, 4, 3, 2, 1, 5, 9, 11, 10, 6, 7, 8, 0] time: 0.000296


### Comparative analysis
The exact method obtains the global optimum. The heuristic is much faster and remains close to optimal, making it more suitable for larger problem sizes.